# Detailed Explanation of CNN Model Training for Digit Recognition

This notebook provides a comprehensive, line-by-line explanation of building, training, and evaluating a Convolutional Neural Network (CNN) to recognize handwritten digits using the MNIST dataset.

## 1. Importing Libraries
First, we import all necessary libraries for data processing, model building, and visualization.

In [ ]:
# Import main deep learning framework (Keras is part of TensorFlow)
from tensorflow import keras

# OpenCV for computer vision and image processing tasks
import cv2

# NumPy for numerical operations and array manipulations
import numpy as np

# Matplotlib for plotting graphs and displaying images
import matplotlib.pyplot as plt

# Seaborn for advanced statistical data visualization (used for the confusion matrix)
import seaborn as sns

# Utility function to convert integer labels into one-hot encoded vectors
from tensorflow.keras.utils import to_categorical

# Callbacks to monitor training, stop early, reduce learning rate, and save the best model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Tool for real-time data augmentation (generating modified images during training)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Layers used to prevent overfitting and stabilize the learning process
from tensorflow.keras.layers import Dropout, BatchNormalization

# Tool from scikit-learn to generate a confusion matrix for evaluation
from sklearn.metrics import confusion_matrix

## 2. Loading the MNIST Dataset
We load the MNIST dataset, which contains 70,000 grayscale images of handwritten digits (0-9).

In [ ]:
# Assign the MNIST dataset module from Keras to a variable
mnist = keras.datasets.mnist

# Load the dataset into memory. It automatically splits into training and testing sets.
# x_train: 60,000 training images. y_train: 60,000 training labels.
# x_test: 10,000 testing images. y_test: 10,000 testing labels.
(x_train, y_train), (x_test, y_test) = mnist.load_data()

## 3. Exploratory Data Analysis (EDA)
Let's visualize the distribution of our training data and see some sample images.

In [ ]:
# Create a bar chart showing how many examples exist for each digit (0 through 9)
sns.countplot(x=y_train)

# Set the title of the plot
plt.title("Distribution of Digits in Training Data")

# Label the x-axis
plt.xlabel("Digit")

# Label the y-axis
plt.ylabel("Count")

# Display the plot
plt.show()

# Set up a grid of 1 row and 6 columns to plot sample images
fig, axes = plt.subplots(1, 6, figsize=(10, 5))

# Loop through the first 6 images in the training set
for i, ax in enumerate(axes):
    # Display the image in grayscale color map
    ax.imshow(x_train[i], cmap='gray')
    
    # Hide the axes (ticks and borders) for a cleaner look
    ax.axis("off")

# Show the sample images
plt.show()

## 4. Data Preprocessing
Neural networks learn better when input data is normalized and shaped correctly. We also need to encode our labels.

In [ ]:
# Pixel values range from 0 to 255. We divide by 255.0 to normalize them to a range of 0.0 to 1.0.
# This helps the neural network learn faster and prevents vanishing/exploding gradients.
x_train = x_train / 255.0
x_test = x_test / 255.0

# CNNs expect inputs in the shape (batch_size, height, width, channels).
# Grayscale images have 1 color channel. We reshape the arrays to add this channel dimension.
# The '-1' tells numpy to figure out the batch size automatically based on the length of the array.
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

# Convert integer labels (e.g., '2') into one-hot encoded vectors (e.g., [0, 0, 1, 0, 0, 0, 0, 0, 0, 0]).
# This is required for categorical crossentropy loss, which compares probabilities for each class.
y_train = to_categorical(y_train, num_classes=10)
y_test = to_categorical(y_test, num_classes=10)

## 5. Data Augmentation
To make our model more robust and prevent overfitting, we artificially expand our dataset by creating slightly modified versions of our images.

In [ ]:
# Initialize the ImageDataGenerator with random transformation rules
datagen = ImageDataGenerator(
    rotation_range=15,       # Randomly rotate images by up to 15 degrees
    zoom_range=0.15,         # Randomly zoom into or out of images by 15%
    width_shift_range=0.15,  # Randomly shift images horizontally by 15% of the width
    height_shift_range=0.15, # Randomly shift images vertically by 15% of the height
    shear_range=0.1,         # Randomly apply shearing transformations
    fill_mode='nearest'      # Fill newly created blank pixels with the nearest original pixel values
)

# Calculate any necessary statistics on the training data (not strictly needed here since we don't use featurewise normalization)
datagen.fit(x_train)

# Generate a single batch of 9 augmented images to visualize the effects
x_batch, y_batch = next(datagen.flow(x_train[:9], y_train[:9], batch_size=9))

# Set up a 3x3 grid to plot the augmented images
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
axes = axes.flatten() # Flatten the 2D array of axes into a 1D array for easy iteration

# Loop through the augmented images and plot them
for i, (img, ax) in enumerate(zip(x_batch, axes)):
    # Reshape the image back to 28x28 for visualization
    ax.imshow(img.reshape(28, 28), cmap='gray')
    
    # Set the title to the correct digit label (reversing the one-hot encoding using argmax)
    ax.set_title(f"Label: {np.argmax(y_batch[i])}")
    
    # Hide the axes
    ax.axis('off')

# Automatically adjust the spacing between subplots
plt.tight_layout()

# Show the augmented images
plt.show()

## 6. Building the CNN Architecture
We define a Sequential model, adding Convolutional layers to extract features and Dense layers to classify the image.

In [ ]:
# Initialize a Sequential model (a linear stack of layers)
model = keras.Sequential([
    # First Convolutional Block
    # Extracts local patterns. 32 filters, 3x3 kernel size, ReLU activation (turns negative values to 0).
    # padding='same' ensures output size matches input size. input_shape defines the input image format.
    keras.layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(28, 28, 1)),
    
    # Normalizes the activations of the previous layer, speeding up training and adding slight regularization.
    BatchNormalization(),
    
    # Another Conv2D layer to learn more complex features based on the first layer's output.
    keras.layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    
    # Reduces the spatial dimensions (width, height) by half, selecting the maximum value in 2x2 blocks.
    keras.layers.MaxPooling2D(2,2),
    
    # Randomly sets 25% of input units to 0 at each step during training to prevent overfitting.
    Dropout(0.25),
    
    # Second Convolutional Block (learning more abstract, higher-level features)
    keras.layers.Conv2D(64, (3,3), activation='relu', padding='same'), # 64 filters this time
    BatchNormalization(),
    keras.layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    keras.layers.MaxPooling2D(2,2),
    Dropout(0.25),
    
    # Classifier Section
    # Flattens the 3D feature maps into a 1D vector so it can be fed into a standard dense (fully connected) layer.
    keras.layers.Flatten(),
    
    # Fully connected layer with 256 neurons to learn complex relationships between extracted features.
    keras.layers.Dense(256, activation='relu'),
    BatchNormalization(),
    
    # High dropout rate (50%) before the final classification to prevent the network from relying on specific neurons.
    Dropout(0.5),
    
    # Output layer: 10 neurons (one for each digit). Softmax activation converts raw scores into probabilities summing to 1.
    keras.layers.Dense(10, activation='softmax')
])

# Print a summary of the model architecture, showing output shapes and the number of parameters.
model.summary()

## 7. Compiling the Model and Callbacks
We define the loss function, optimizer, and callbacks that will control the training process.

In [ ]:
# Compile the model, preparing it for training
model.compile(
    # Adam optimizer adapts the learning rate during training for faster and better convergence
    optimizer='adam',
    # Categorical crossentropy is the standard loss function for multi-class classification problems
    loss='categorical_crossentropy',
    # We want to track accuracy during training
    metrics=['accuracy']
)

# Callback 1: Early Stopping
# Stops training if the validation loss doesn’t improve for 10 consecutive epochs, preventing overfitting and saving time.
early_stop = EarlyStopping(
    monitor='val_loss',         # Metric to watch
    patience=10,                # How many epochs to wait before stopping
    restore_best_weights=True,  # Restores the model weights from the epoch with the best validation loss
    verbose=1                   # Print a message when early stopping triggers
)

# Callback 2: Reduce Learning Rate on Plateau
# Reduces the learning rate by a factor of 0.2 if validation loss doesn’t improve for 5 epochs.
# A smaller learning rate helps the model converge into the minimum loss without jumping over it.
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',         # Metric to watch
    factor=0.2,                 # Multiply the learning rate by this factor
    patience=5,                 # How many epochs to wait before reducing
    min_lr=1e-6,                # Don't let the learning rate go below this value
    verbose=1                   # Print a message when learning rate is reduced
)

# Callback 3: Model Checkpoint
# Saves the best version of the model (highest validation accuracy) to disk during training.
checkpoint = ModelCheckpoint(
    'best_digit_moodel.h5',     # File name to save to
    monitor='val_accuracy',     # Metric to watch
    save_best_only=True,        # Only overwrite if the model is strictly better than the previously saved one
    verbose=1                   # Print a message when saving
)

## 8. Training the Model
We train the model using our augmented data generators and the specified callbacks.

In [ ]:
# Start the training process
history = model.fit(
    # Use datagen.flow() to provide augmented batches of training data dynamically
    datagen.flow(x_train, y_train, batch_size=64),
    # Set maximum number of epochs to 30 (early stopping might stop it sooner)
    epochs=30,
    # Use the test set to validate the model's performance at the end of each epoch
    validation_data=(x_test, y_test),
    # Pass our three callbacks to monitor and adjust the training process
    callbacks=[early_stop, reduce_lr, checkpoint],
    # Print progress bars during training
    verbose=1
)

## 9. Visualizing Training History
We plot the accuracy and loss curves for both the training and validation sets to check for overfitting and evaluate training success.

In [ ]:
# Set up a figure with 2 subplots (1 row, 2 columns)
plt.figure(figsize=(12, 4))

# First subplot for Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'])      # Plot training accuracy
plt.plot(history.history['val_accuracy'])  # Plot validation accuracy
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='lower right') # Add a legend

# Second subplot for Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])          # Plot training loss
plt.plot(history.history['val_loss'])      # Plot validation loss
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper right') # Add a legend

# Adjust layout and display the plots
plt.tight_layout()
plt.show()

## 10. Evaluating the Best Model
We load the best model saved during training and test it on the unseen test dataset.

In [ ]:
# Import load_model utility
from tensorflow.keras.models import load_model

# Load the model weights that achieved the highest validation accuracy
best_model = load_model('best_digit_moodel.h5')

# Evaluate the model on the test dataset (which wasn't used for training or validation during model selection)
test_loss, test_acc = best_model.evaluate(x_test, y_test, verbose=0)

# Print the final test metrics
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")

## 11. Confusion Matrix
A confusion matrix helps us understand which digits the model frequently confuses with each other.

In [ ]:
# Get model predictions for the entire test set
y_pred = best_model.predict(x_test)

# Convert the predicted probabilities into final class labels (0-9)
y_pred_classes = np.argmax(y_pred, axis=1)

# Convert the one-hot encoded true labels back to class labels (0-9)
y_true = np.argmax(y_test, axis=1)

# Generate the confusion matrix using scikit-learn
cm = confusion_matrix(y_true, y_pred_classes)

# Plot the confusion matrix using Seaborn's heatmap
plt.figure(figsize=(10, 8))
# annot=True displays the numbers in the boxes. fmt='d' formats them as integers. cmap='Blues' sets the color scheme.
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

# Print out specific pairs of digits that were confused more than 5 times
print("Most confused digit pairs:")
for i in range(10): # Iterate over true labels
    for j in range(10): # Iterate over predicted labels
        if i != j and cm[i, j] > 5:
            print(f"True {i}, Predicted {j}: {cm[i, j]} instances")

## 12. Visualizing Random Predictions
Let's randomly select a few images from the test set and see how our model performs visually.

In [ ]:
# Create a figure for plotting
plt.figure(figsize=(15, 6))

# Plot 10 random examples
for i in range(10):
    # Set up a 2x5 grid of subplots
    plt.subplot(2, 5, i+1)
    
    # Pick a random index from the test set
    index = np.random.randint(0, len(x_test))
    
    # Reshape the image back to 28x28 for displaying
    img = x_test[index].reshape(28, 28)
    
    # Get the true label
    true_label = np.argmax(y_test[index])
    
    # Get the model's prediction for this specific image
    # Note: predict() expects a batch, so we reshape it to (1, 28, 28, 1)
    predicted = np.argmax(best_model.predict(x_test[index].reshape(1, 28, 28, 1)))
    
    # Display the image
    plt.imshow(img, cmap='gray')
    
    # Set the title to show True and Predicted labels
    plt.title(f"True: {true_label}\nPred: {predicted}")
    
    # Hide axes
    plt.axis('off')

# Adjust layout and show the plots
plt.tight_layout()
plt.show()

## 13. Saving the Final Model
Finally, we save the completed model for future use in our web application.

In [ ]:
# Save the fully trained and tested model to an HDF5 file
# This is the file that will be loaded by our Flask backend (app.py) to make predictions in real-time.
best_model.save("digit_recognizer_improved110.h5")

# Print confirmation
print("Model saved successfully as 'digit_recognizer_improved110.h5'")